# 02 - Evaluate Generated Attacks Against Gemini

This notebook evaluates the attacks generated by `01_create_expand_dataset.ipynb`.

It uses Promptfoo with the local Gemini REST provider in this repo:

```text
providers/gemini_rest_provider.js
```

The provider reads your Gemini API key from:

```text
GOOGLE_API_KEY
```

Do not paste API keys into notebooks, source files, screenshots, or chat logs.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "finance_redteam").exists():
    candidates = [p for p in Path.cwd().parents if (p / "src" / "finance_redteam").exists()]
    if not candidates:
        raise RuntimeError("Could not find project root. Open this notebook from finance-llm-redteam-benchmark.")
    PROJECT_ROOT = candidates[0]

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

## 2. Check Dataset and Promptfoo Config

This notebook expects the first notebook to have created the JSONL and YAML files.

In [ ]:
ATTACKS_JSONL = Path("data/exports/finance_redteam_attacks.jsonl")
PROMPTFOO_YAML = Path("data/exports/promptfoo_tests.yaml")

if not ATTACKS_JSONL.exists():
    raise FileNotFoundError(f"Missing {ATTACKS_JSONL}. Run notebook 01 first.")
if not PROMPTFOO_YAML.exists():
    raise FileNotFoundError(f"Missing {PROMPTFOO_YAML}. Run notebook 01 first.")

record_count = sum(1 for _ in ATTACKS_JSONL.open("r", encoding="utf-8"))
print(f"Attack records: {record_count}")
print(f"Promptfoo config: {PROMPTFOO_YAML}")
print(PROMPTFOO_YAML.read_text(encoding="utf-8").splitlines()[:8])

## 3. Validate Dataset Locally

This checks schema and safety rules before model evaluation.

In [ ]:
from finance_redteam.exporters import load_jsonl
from finance_redteam.validator import validate_records

records = load_jsonl(ATTACKS_JSONL)
result = validate_records(records)

print(f"Validation passed: {result.valid}")
if result.review_warnings:
    print("Review warnings:")
    for warning in result.review_warnings[:20]:
        print("-", warning)
if not result.valid:
    for error in result.errors[:50]:
        print("ERROR:", error)
    raise RuntimeError("Dataset validation failed")

## 4. Set Gemini Key Safely

Recommended: set `GOOGLE_API_KEY` in your terminal before starting Jupyter or VS Code.

Example terminal command:

```bash
export GOOGLE_API_KEY="your_key_here"
```

If you set it inside the notebook, do not save or share the notebook with the key visible.

In [ ]:
if os.getenv("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY is set in this notebook process.")
else:
    print("GOOGLE_API_KEY is not set. Set it before running Gemini evaluation cells.")
    print("Example: os.environ['GOOGLE_API_KEY'] = 'your_key_here'  # avoid saving this cell with the real key")

## 5. List Available Gemini Models

If you get a 404 saying a model is not found or does not support `generateContent`, run this cell and choose one of the listed model names.

The local provider uses `GEMINI_MODEL`. If not set, it defaults to `gemini-2.0-flash`.

In [ ]:
list_models = subprocess.run(
    [sys.executable, "scripts/list_gemini_models.py"],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)
print(list_models.stdout)
if list_models.stderr:
    print(list_models.stderr)
if list_models.returncode != 0:
    raise RuntimeError("Could not list Gemini models. Check GOOGLE_API_KEY first.")

## 6. Test Gemini Key With One Harmless Request

This calls `scripts/test_gemini_key.py`. It does not print your key.

In [ ]:
completed = subprocess.run(
    [sys.executable, "scripts/test_gemini_key.py"],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError("Gemini key test failed. Fix the key/network issue before running Promptfoo.")

## 7. Promptfoo Smoke Test

Run the first attack only. This is the safest first evaluation step and helps avoid quota problems.

In [ ]:
def run_command(command: list[str], timeout: int = 600) -> subprocess.CompletedProcess[str]:
    env = os.environ.copy()
    env["HOME"] = str(PROJECT_ROOT)
    env["PROMPTFOO_DISABLE_TELEMETRY"] = "1"
    print("Running:", " ".join(command))
    completed = subprocess.run(command, text=True, capture_output=True, env=env, timeout=timeout)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    return completed

smoke = run_command([
    "npx", "promptfoo", "eval",
    "-c", str(PROMPTFOO_YAML),
    "--filter-first-n", "1",
    "--max-concurrency", "1",
    "--delay", "5000",
    "--no-cache",
    "--no-progress-bar",
], timeout=600)

if smoke.returncode != 0:
    raise RuntimeError("Promptfoo smoke test failed. Review the output above.")

## 8. Full Promptfoo Evaluation

Run this only after the smoke test succeeds.

The output is written to `data/exports/promptfoo_results.json`.

In [ ]:
RUN_FULL_EVAL = False  # Change to True after smoke test succeeds.

if RUN_FULL_EVAL:
    full = run_command([
        "npx", "promptfoo", "eval",
        "-c", str(PROMPTFOO_YAML),
        "--max-concurrency", "1",
        "--delay", "1000",
        "--no-cache",
        "--no-progress-bar",
        "--output", "data/exports/promptfoo_results.json",
    ], timeout=7200)
    if full.returncode != 0:
        raise RuntimeError("Full Promptfoo evaluation failed. Review the output above.")
else:
    print("Full eval skipped. Set RUN_FULL_EVAL = True when you are ready.")

## 9. Retry Errors Only

Use this if the full run has temporary Gemini/API errors.

In [ ]:
RUN_RETRY_ERRORS = False

if RUN_RETRY_ERRORS:
    retry = run_command([
        "npx", "promptfoo", "eval",
        "-c", str(PROMPTFOO_YAML),
        "--retry-errors",
        "--max-concurrency", "1",
        "--delay", "1000",
        "--no-cache",
        "--no-progress-bar",
    ], timeout=7200)
    if retry.returncode != 0:
        raise RuntimeError("Retry-errors run failed. Review the output above.")
else:
    print("Retry-errors skipped. Set RUN_RETRY_ERRORS = True if needed.")

## 10. Stop A Stuck Eval

If a Promptfoo process gets stuck, run this cell. It terminates Promptfoo processes for this machine user.

In [ ]:
STOP_STUCK_PROMPTFOO = False  # Set True only when you need to stop an active eval.

if STOP_STUCK_PROMPTFOO:
    subprocess.run(["pkill", "-f", "promptfoo"], check=False)
    subprocess.run(["pkill", "-f", "node.*promptfoo"], check=False)
    print("Requested Promptfoo process stop.")
else:
    print("Stop skipped. Set STOP_STUCK_PROMPTFOO = True to terminate Promptfoo.")

## 11. Optional Garak / DeepTeam Evaluation Notes

In this repo, DeepTeam and Garak are primarily used to expand the dataset in notebook 01.

For evaluation:

- Promptfoo runs the generated attack prompts against Gemini.
- Garak can also be run separately as a scanner if you configure a supported target model.
- DeepTeam can be used for more advanced adversarial evaluation if you wire its runtime evaluator to your target model.

The current end-to-end path is intentionally simple and reliable:

```text
Notebook 01 creates attacks -> Notebook 02 evaluates attacks with Promptfoo against Gemini
```